In [2]:
!pip install -q gradio transformers ftfy
!pip install -q git+https://github.com/openai/CLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
"""
Few-Shot Learning with CLIP and BLIP — Interactive Demo
IUT CSE Semester 6 — CVLab

Run with:
    pip install gradio transformers torch torchvision ftfy
    pip install git+https://github.com/openai/CLIP.git
    python gradio_demo.py

Or on Colab:
    !pip install -q gradio transformers ftfy
    !pip install -q git+https://github.com/openai/CLIP.git
    then run the cell
"""

import gradio as gr
import torch
import clip
import numpy as np
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
from transformers import BlipForImageTextRetrieval
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')

# ── Device ─────────────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading models on {DEVICE}...')

# ── Load CLIP ──────────────────────────────────────────────────────────────
print('Loading CLIP ViT-B/32...')
clip_model, clip_preprocess = clip.load('ViT-B/32', device=DEVICE)
clip_model.eval()
print('CLIP loaded.')

# ── Load BLIP ──────────────────────────────────────────────────────────────
print('Loading BLIP captioning model...')
blip_processor = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
blip_caption_model = BlipForConditionalGeneration.from_pretrained(
    'Salesforce/blip-image-captioning-base',
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32
).to(DEVICE)
blip_caption_model.eval()

print('Loading BLIP ITM model...')
blip_itm_model = BlipForImageTextRetrieval.from_pretrained(
    'Salesforce/blip-itm-base-coco',
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32
).to(DEVICE)
blip_itm_model.eval()
print('All models loaded. Starting Gradio...')


Loading models on cuda...
Loading CLIP ViT-B/32...


100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 101MiB/s]


CLIP loaded.
Loading BLIP captioning model...


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  895MB            

model.safetensors: downloading bytes:           |  0.00B            

All models loaded. Starting Gradio...


In [4]:
# ═══════════════════════════════════════════════════════════════════════════
# TAB 1: CLIP Zero-Shot Classification
# ═══════════════════════════════════════════════════════════════════════════

def clip_zero_shot(image, class_names_text, prompt_template):
    """
    Zero-shot classify an image against user-defined class names.
    No training. No examples. Pure CLIP zero-shot.

    Arguments:
        image           -- PIL Image uploaded by user
        class_names_text -- comma-separated class names
        prompt_template  -- e.g. "a photo of a {}"
    """
    if image is None:
        return "Please upload an image.", None

    # Parse class names
    class_names = [c.strip() for c in class_names_text.split(',') if c.strip()]
    if len(class_names) < 2:
        return "Please enter at least 2 class names separated by commas.", None

    # Build text prompts using template
    prompts = [prompt_template.replace('{}', name) for name in class_names]

    # Encode image
    img_input = clip_preprocess(image).unsqueeze(0).to(DEVICE)
    text_tokens = clip.tokenize(prompts).to(DEVICE)

    with torch.no_grad():
        image_features = clip_model.encode_image(img_input)
        text_features  = clip_model.encode_text(text_tokens)

        # L2 normalise
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features  = text_features  / text_features.norm(dim=-1, keepdim=True)

        # Cosine similarity * temperature
        logits = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        scores = logits[0].cpu().numpy()

    # Build result
    pred_idx   = scores.argmax()
    pred_class = class_names[pred_idx]
    confidence = scores[pred_idx] * 100

    result_text = f"**Prediction: {pred_class}** ({confidence:.1f}% confidence)\n\n"
    result_text += "**All scores:**\n"
    for name, score in sorted(zip(class_names, scores), key=lambda x: -x[1]):
        bar = '█' * int(score * 30)
        result_text += f"  {name:20s} {score*100:5.1f}%  {bar}\n"

    result_text += f"\n**How it works:**\n"
    result_text += f"CLIP encoded the image into a 512-dim vector.\n"
    result_text += f"It also encoded {len(class_names)} text prompts: {prompts[:2]}...\n"
    result_text += f"The prediction is the class whose text embedding is most similar to the image embedding.\n"
    result_text += f"**No training was used. This is pure zero-shot classification.**"

    return result_text

In [5]:
# ═══════════════════════════════════════════════════════════════════════════
# TAB 2: Few-Shot Prototype Classification
# ═══════════════════════════════════════════════════════════════════════════

# Storage for support set
support_store = {}

def add_support_image(image, class_name):
    """Add one image to the support set for a class."""
    if image is None:
        return "Please upload an image.", format_support_status()
    if not class_name.strip():
        return "Please enter a class name.", format_support_status()

    cls = class_name.strip().lower()

    # Encode with CLIP
    img_input = clip_preprocess(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        feat = clip_model.encode_image(img_input)
        feat = feat / feat.norm(dim=-1, keepdim=True)

    if cls not in support_store:
        support_store[cls] = []
    support_store[cls].append(feat.cpu())

    count = len(support_store[cls])
    return (
        f"✅ Added image #{count} to class '{cls}'.",
        format_support_status()
    )

def format_support_status():
    """Format current support set status."""
    if not support_store:
        return "Support set is empty. Add some images above."
    lines = ["**Current support set:**"]
    for cls, feats in support_store.items():
        lines.append(f"  - {cls}: {len(feats)} image(s)")
    return "\n".join(lines)

def clear_support():
    """Clear the support set."""
    support_store.clear()
    return "Support set cleared.", format_support_status()

def few_shot_predict(query_image):
    """
    Classify query image using prototype matching.

    ProtoNet algorithm (Snell et al., 2017) applied to CLIP features:
    1. Compute prototype for each class = mean of support embeddings
    2. Encode query image
    3. Predict class with nearest prototype (by cosine similarity)
    """
    if query_image is None:
        return "Please upload a query image."
    if len(support_store) < 2:
        return "Please add images for at least 2 classes to the support set first."

    # Compute prototypes
    prototypes = {}
    for cls, feats in support_store.items():
        stacked = torch.cat(feats, dim=0)          # (K, 512)
        prototype = stacked.mean(dim=0, keepdim=True)  # (1, 512)
        prototype = prototype / prototype.norm(dim=-1, keepdim=True)
        prototypes[cls] = prototype.to(DEVICE)

    # Encode query
    img_input = clip_preprocess(query_image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        query_feat = clip_model.encode_image(img_input)
        query_feat = query_feat / query_feat.norm(dim=-1, keepdim=True)

    # Compute similarities to all prototypes
    class_names = list(prototypes.keys())
    proto_matrix = torch.cat([prototypes[c] for c in class_names], dim=0)  # (N, 512)
    sims = (query_feat @ proto_matrix.T)[0]                                 # (N,)
    probs = F.softmax(sims * 20, dim=0).cpu().numpy()                      # sharpen

    pred_idx   = probs.argmax()
    pred_class = class_names[pred_idx]
    confidence = probs[pred_idx] * 100

    result = f"**Prediction: {pred_class}** ({confidence:.1f}% confidence)\n\n"
    result += "**Similarity to each class prototype:**\n"
    for name, prob in sorted(zip(class_names, probs), key=lambda x: -x[1]):
        bar = '█' * int(prob * 30)
        result += f"  {name:20s} {prob*100:5.1f}%  {bar}\n"

    k_total = sum(len(v) for v in support_store.values())
    result += f"\n**How it works (ProtoNet + CLIP):**\n"
    result += f"You provided {k_total} support images across {len(class_names)} classes.\n"
    result += f"Each class prototype = mean CLIP embedding of its support images.\n"
    result += f"Query is classified to the nearest prototype by cosine similarity.\n"
    result += f"**This is {len(class_names)}-way {k_total//len(class_names) if len(class_names)>0 else 1}-shot classification.**"

    return result


In [6]:
# ═══════════════════════════════════════════════════════════════════════════
# TAB 3: BLIP Image Captioning
# ═══════════════════════════════════════════════════════════════════════════

def blip_caption(image, max_tokens, num_beams):
    """Generate a caption for an image using BLIP."""
    if image is None:
        return "Please upload an image."

    inputs = blip_processor(
        images=image,
        return_tensors='pt'
    ).to(DEVICE, torch.float16 if DEVICE == 'cuda' else torch.float32)

    with torch.no_grad():
        output_ids = blip_caption_model.generate(
            **inputs,
            max_new_tokens=int(max_tokens),
            num_beams=int(num_beams),
            early_stopping=True
        )

    caption = blip_processor.decode(output_ids[0], skip_special_tokens=True)

    result  = f"**Generated Caption:**\n\n> {caption}\n\n"
    result += f"**Settings:** max_tokens={int(max_tokens)}, beam_search_width={int(num_beams)}\n\n"
    result += "**How BLIP works:**\n"
    result += "BLIP uses a multimodal encoder-decoder architecture.\n"
    result += "The image is encoded by a Vision Transformer.\n"
    result += "The decoder generates caption tokens autoregressively.\n"
    result += "Beam search explores multiple candidate captions and picks the best.\n"
    result += "**No task-specific training — this is zero-shot captioning from BLIP's pre-training.**"

    return result

In [7]:
# ═══════════════════════════════════════════════════════════════════════════
# TAB 4: BLIP Image-Text Matching
# ═══════════════════════════════════════════════════════════════════════════

def blip_match(image, text_descriptions):
    """Score how well each text description matches the image."""
    if image is None:
        return "Please upload an image."

    descriptions = [d.strip() for d in text_descriptions.strip().split('\n') if d.strip()]
    if not descriptions:
        return "Please enter at least one text description."

    results = []
    for desc in descriptions:
        inputs = blip_processor(
            images=image,
            text=desc,
            return_tensors='pt'
        ).to(DEVICE, torch.float16 if DEVICE == 'cuda' else torch.float32)

        with torch.no_grad():
            output = blip_itm_model(**inputs, use_itm_head=True)
            score  = torch.nn.functional.softmax(output.itm_score, dim=1)
            match_prob = score[0][1].item()

        results.append((desc, match_prob))

    results.sort(key=lambda x: -x[1])

    output = "**Image-Text Match Scores** (higher = better match):\n\n"
    for desc, prob in results:
        bar   = '█' * int(prob * 40)
        emoji = '✅' if prob > 0.7 else '⚠️' if prob > 0.4 else '❌'
        output += f"{emoji} **{prob*100:.1f}%** — {desc}\n    {bar}\n\n"

    output += "**How it works:**\n"
    output += "BLIP's Image-Text Matching (ITM) head scores whether a text correctly describes an image.\n"
    output += "Score > 70% = strong match. 40-70% = uncertain. < 40% = poor match.\n"
    output += "This is used in BLIP's training pipeline to filter noisy web-scraped image-text pairs."

    return output

In [9]:
# ═══════════════════════════════════════════════════════════════════════════
# TAB 5: About / How It Works
# ═══════════════════════════════════════════════════════════════════════════

ABOUT_TEXT = """
# Few-Shot Learning with CLIP and BLIP
## IUT CSE Semester 6 — CVLab Design Project

---

## What is Few-Shot Learning?

Few-shot learning (FSL) is the ability to classify or recognise new categories
from only a small number of labeled examples — sometimes just 1 (one-shot) or 0 (zero-shot).

Humans do this naturally: seeing one photo of an unfamiliar animal is enough to recognise it later.
Standard deep learning needs thousands of examples. FSL bridges this gap.

---

## CLIP (Radford et al., ICML 2021)

**Architecture:**
- Image Encoder: Vision Transformer (ViT) — splits image into patches, processes with self-attention
- Text Encoder: Causal Transformer (GPT-style) — processes tokenised text
- Both encoders map to a **shared 512-dimensional embedding space**

**Training:**
- 400 million image-text pairs from the internet
- Symmetric contrastive loss: pushes matching pairs together, non-matching pairs apart
- S = Image_features @ Text_features.T * exp(temperature)
- L = (CrossEntropy(S, labels) + CrossEntropy(S.T, labels)) / 2

**Zero-shot inference:**
- Encode class names as text: "a photo of a cat", "a photo of a dog", ...
- Encode query image
- Predict class with highest cosine similarity to image embedding
- **No training on target classes needed!**

---

## BLIP (Li et al., ICML 2022)

**Extends CLIP with generative capability:**
- Image captioning: generates natural language descriptions of images
- Image-text matching: scores how well text describes an image
- Visual Question Answering (VQA)

**Key innovation — Bootstrapping:**
- Generates synthetic captions for noisy web images using a captioner
- Filters them using an ITM module
- Self-improves training data quality automatically

---

## This Demo

| Tab | What it shows | FSL concept |
|-----|---------------|-------------|
| Zero-Shot CLIP | Classify with text only | Zero-shot (K=0) |
| Few-Shot Prototype | Classify from examples | N-way K-shot |
| BLIP Caption | Generate image description | Generative VLM |
| BLIP Matching | Score text-image pairs | Cross-modal alignment |

---

## Paper References
- Radford et al. (2021). Learning Transferable Visual Models From Natural Language Supervision. ICML.
- Li et al. (2022). BLIP: Bootstrapping Language-Image Pre-training. ICML.
- Snell et al. (2017). Prototypical Networks for Few-Shot Learning. NeurIPS.
- Wang et al. (2020). Generalizing from a Few Examples: A Survey on Few-Shot Learning. ACM Comput. Surv.

---

*Islamic University of Technology — Department of CSE*
*Supervisors: Bakhtiar Sir & Maria Ma'am*
"""


# ═══════════════════════════════════════════════════════════════════════════
# Build Gradio Interface
# ═══════════════════════════════════════════════════════════════════════════

with gr.Blocks(
    title='Few-Shot Learning Demo — IUT CVLab',
    theme=gr.themes.Soft(primary_hue='blue'),
    css='.gradio-container { max-width: 900px; margin: auto; }'
) as demo:

    gr.Markdown("""
    # 🎓 Few-Shot Learning with CLIP and BLIP
    **IUT CSE Semester 6 — CVLab Design Project**
    *Supervisors: Bakhtiar Sir & Maria Ma'am*

    Demonstrating zero-shot and few-shot image classification, image captioning, and image-text matching
    using CLIP (Radford et al., 2021) and BLIP (Li et al., 2022).
    """)

    with gr.Tabs():

        # ── Tab 1: Zero-Shot CLIP ─────────────────────────────────────
        with gr.Tab('🔍 Zero-Shot CLIP'):
            gr.Markdown("""
            ## CLIP Zero-Shot Classification
            Upload any image. Enter class names. CLIP classifies without any training examples.
            This directly demonstrates **zero-shot learning** — the extreme case of few-shot learning where K=0.
            """)

            with gr.Row():
                with gr.Column(scale=1):
                    zs_image = gr.Image(type='pil', label='Upload Image')
                    zs_classes = gr.Textbox(
                        label='Class Names (comma-separated)',
                        value='cat, dog, car, bird, airplane',
                        placeholder='cat, dog, car, airplane, ship'
                    )
                    zs_template = gr.Textbox(
                        label='Prompt Template (use {} for class name)',
                        value='a photo of a {}',
                        placeholder='a photo of a {}'
                    )
                    zs_btn = gr.Button('🚀 Classify (Zero-Shot)', variant='primary')

                with gr.Column(scale=1):
                    zs_output = gr.Markdown(label='Result')

            zs_btn.click(
                fn=clip_zero_shot,
                inputs=[zs_image, zs_classes, zs_template],
                outputs=[zs_output]
            )

            gr.Examples(
                examples=[
                    [None, 'cat, dog, car, airplane, ship, horse', 'a photo of a {}'],
                    [None, 'happy person, sad person, angry person', 'a {} face'],
                    [None, 'pizza, burger, sushi, pasta', 'a photo of {} food'],
                ],
                inputs=[zs_image, zs_classes, zs_template]
            )

        # ── Tab 2: Few-Shot Prototype ─────────────────────────────────
        with gr.Tab('🎯 Few-Shot Prototype (N-way K-shot)'):
            gr.Markdown("""
            ## Few-Shot Prototype Classification (ProtoNet + CLIP)
            **Step 1:** Add support images with class labels (these are your K-shot examples).
            **Step 2:** Upload a query image to classify.

            This implements **Prototypical Networks (Snell et al., 2017)** on top of CLIP features:
            each class prototype = mean embedding of its support images.
            """)

            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### Step 1: Build Support Set")
                    sup_image      = gr.Image(type='pil', label='Support Image')
                    sup_class_name = gr.Textbox(
                        label='Class Name for this image',
                        placeholder='e.g. cat'
                    )
                    with gr.Row():
                        add_btn   = gr.Button('➕ Add to Support Set', variant='primary')
                        clear_btn = gr.Button('🗑️ Clear Support Set', variant='secondary')
                    add_status    = gr.Markdown()
                    support_status = gr.Markdown('Support set is empty.')

                with gr.Column(scale=1):
                    gr.Markdown("### Step 2: Classify Query Image")
                    query_image = gr.Image(type='pil', label='Query Image')
                    predict_btn = gr.Button('🎯 Predict Class', variant='primary')
                    fs_output   = gr.Markdown()

            add_btn.click(
                fn=add_support_image,
                inputs=[sup_image, sup_class_name],
                outputs=[add_status, support_status]
            )
            clear_btn.click(
                fn=clear_support,
                inputs=[],
                outputs=[add_status, support_status]
            )
            predict_btn.click(
                fn=few_shot_predict,
                inputs=[query_image],
                outputs=[fs_output]
            )

            gr.Markdown("""
            **Example workflow (5-way 1-shot):**
            1. Add 1 image of a cat, label it "cat"
            2. Add 1 image of a dog, label it "dog"
            3. Add 1 image of a car, label it "car"
            4. Upload an unknown image as query
            5. Click Predict — CLIP classifies by nearest prototype
            """)

        # ── Tab 3: BLIP Caption ───────────────────────────────────────
        with gr.Tab('📝 BLIP Captioning'):
            gr.Markdown("""
            ## BLIP Image Captioning
            Upload an image and BLIP generates a natural language description.
            This demonstrates **generative vision-language understanding** — BLIP's key capability beyond CLIP.
            No task-specific training. Pure zero-shot captioning from BLIP's pre-training.
            """)

            with gr.Row():
                with gr.Column(scale=1):
                    cap_image    = gr.Image(type='pil', label='Upload Image')
                    max_tokens   = gr.Slider(10, 60, value=30, step=5, label='Max Caption Length (tokens)')
                    num_beams    = gr.Slider(1, 8, value=4, step=1, label='Beam Search Width')
                    cap_btn      = gr.Button('📝 Generate Caption', variant='primary')

                with gr.Column(scale=1):
                    cap_output = gr.Markdown()

            cap_btn.click(
                fn=blip_caption,
                inputs=[cap_image, max_tokens, num_beams],
                outputs=[cap_output]
            )

            gr.Markdown("""
            **Beam search width explained:**
            - Width=1: greedy decoding (fastest, less accurate)
            - Width=4: standard beam search (good balance)
            - Width=8: wider search (slower, sometimes better)
            """)

        # ── Tab 4: BLIP Matching ──────────────────────────────────────
        with gr.Tab('🔗 BLIP Image-Text Matching'):
            gr.Markdown("""
            ## BLIP Image-Text Matching (ITM)
            Upload an image and enter several text descriptions (one per line).
            BLIP scores how well each description matches the image.
            This is how BLIP filters noisy training data during its bootstrapping pre-training phase.
            """)

            with gr.Row():
                with gr.Column(scale=1):
                    itm_image = gr.Image(type='pil', label='Upload Image')
                    itm_texts = gr.Textbox(
                        label='Text Descriptions (one per line)',
                        lines=6,
                        value='a photo of a cat\na photo of a dog\na photo of a car\nan image of an animal\na picture of food'
                    )
                    itm_btn = gr.Button('🔗 Score Matches', variant='primary')

                with gr.Column(scale=1):
                    itm_output = gr.Markdown()

            itm_btn.click(
                fn=blip_match,
                inputs=[itm_image, itm_texts],
                outputs=[itm_output]
            )

        # ── Tab 5: About ──────────────────────────────────────────────
        with gr.Tab('📚 About / How It Works'):
            gr.Markdown(ABOUT_TEXT)

    gr.Markdown("""
    ---
    *Islamic University of Technology — CSE Department — CVLab*
    *CLIP: Radford et al. (2021) | BLIP: Li et al. (2022) | ProtoNet: Snell et al. (2017)*
    """)

# ── Launch ─────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    demo.launch(
        share=True,       # Creates a public URL for presentations
        debug=False,
        show_error=True
    )

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://093125d9fa328300df.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
